# V2 Academy (사교육) Features

## 목적

학군 Feature의 마지막 축인 **사교육 환경(학원)**을 단지별·시점별로 생성하고,
02b의 학교 Feature와 합쳐 최종 학교환경 파일 `school_environment_v2.parquet`을 만든다.

## 데이터와 역할

| 데이터 | 시점 | 용도 |
|---|---|---|
| 소상공인시장진흥공단 상가(상권)정보 (서울, 2020-03 ~ 2026-03 분기 25개) | 분기 스냅샷 | **모델 Feature** (학원 수·밀도), 학원가 정의 |
| 서울시 학원 교습소정보 (현재 개원 중인 학원·교습소) | 현재 1개 시점 | **웹앱 표시용** (입시·보습 학원 수·정원), 상가정보 검증 |

학원 교습소정보는 폐원한 학원이 빠진 현재 명부라 과거 시점 Feature로 쓰면 생존 편향이 생긴다.
따라서 모델 Feature에는 사용하지 않는다.

## 시점 원칙

- 상가정보 스냅샷은 기준월 말일 이후 약 5주 뒤 공개된다. (예: 2026-06-30 기준 파일 → 2026-08-05 공개)
  → **기준월 + 3개월부터 사용**한다.
- **2023-03 ~ 2024-09 7개 분기는 수집 결손**이다. (입시·교과학원이 약 15,000 → 6,000개로 감소 후 2024-12 복구,
  사라진 업소 1만여 개는 다른 업종으로 재분류된 것이 아니라 데이터에서 누락)
- **2025-03 분기는 좌표 오류**다. 전 업소 좌표가 위도 +0.9°, 경도 +1.3°(강원도 위치)로 밀려 있고
  밀린 정도가 업소마다 ±200m씩 달라 일괄 보정할 수 없다.
- 결손·좌표 오류 분기는 사용하지 않고 직전 정상 스냅샷을 사용한다. 보간은 미래 스냅샷을 사용하므로 하지 않는다.
- 학원가 위치는 학습 기간(2021-07~) 이전인 **2020-12 스냅샷**으로 정의한다.

In [1]:
# ============================================================
# 프로젝트 경로 설정
# - 상가정보 원본(약 6.9GB)은 data/external/상가정보/에 두고 git에서 제외한다.
# - 교육 업종만 추출한 결과를 processed/academy/에 저장해 재사용한다.
# ============================================================

import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

from dotenv import load_dotenv
from sklearn.cluster import DBSCAN
from sklearn.neighbors import BallTree


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "data").exists():
            return path

    raise FileNotFoundError(
        "프로젝트 루트에서 data 폴더를 찾을 수 없습니다."
    )


PROJECT_DIR = find_project_root()

EXTERNAL_DIR = PROJECT_DIR / "data" / "external"
STORE_DIR = EXTERNAL_DIR / "상가정보"

PROCESSED_DIR = (
    PROJECT_DIR
    / "data"
    / "v2_5year_seoul"
    / "processed"
)

ACADEMY_DIR = PROCESSED_DIR / "academy"
ACADEMY_DIR.mkdir(parents=True, exist_ok=True)

EARTH_RADIUS_M = 6_371_000


print("STORE_DIR:", STORE_DIR)
print("상가정보 파일 수:", len(list(STORE_DIR.glob("*.csv"))))

STORE_DIR: C:\Users\uk\Apartment price\data\external\상가정보
상가정보 파일 수: 25


In [2]:
# ============================================================
# 상가정보 25개 분기에서 교육 업종(P1)만 추출
# - 파일명 끝 YYYYMM을 스냅샷 기준월로 사용한다.
# - 추출 결과를 Parquet으로 저장해 이후에는 원본 CSV를 다시 읽지 않는다.
# ============================================================

STORE_COLS = [
    "상가업소번호",
    "상호명",
    "상권업종대분류코드",
    "상권업종중분류코드",
    "상권업종소분류코드",
    "상권업종소분류명",
    "시군구명",
    "행정동명",
    "경도",
    "위도",
]

STORE_EDU_PATH = ACADEMY_DIR / "store_education_2020_2026.parquet"


if STORE_EDU_PATH.exists():

    store_edu = pd.read_parquet(STORE_EDU_PATH)

else:

    frames = []

    for path in sorted(STORE_DIR.glob("*.csv")):

        snapshot = pd.read_csv(
            path,
            encoding="utf-8-sig",
            usecols=STORE_COLS,
            dtype=str,
        )

        edu = snapshot[
            snapshot["상권업종대분류코드"] == "P1"
        ].copy()

        edu["기준월"] = path.stem.split("_")[-1]
        edu["전체업소수"] = len(snapshot)

        frames.append(edu)

    store_edu = pd.concat(frames, ignore_index=True)
    store_edu[["경도", "위도"]] = store_edu[["경도", "위도"]].astype(float)

    store_edu.to_parquet(STORE_EDU_PATH, index=False)


print("교육 업종 행 수:", f"{len(store_edu):,}")
print("스냅샷 수:", store_edu["기준월"].nunique())
print("좌표 결측:", store_edu[["경도", "위도"]].isna().sum().sum())

교육 업종 행 수: 910,880
스냅샷 수: 25
좌표 결측: 0


In [3]:
# ============================================================
# 분기별 학원 수 추이와 수집 결손 분기 탐지
# - 입시·교과학원(P10501) 수가 전체 분기 중앙값의 70% 미만이면 결손으로 판단한다.
# - 결손 분기는 전체 상가 수도 함께 줄어들어 실제 폐업이 아닌 수집 범위 문제로 본다.
# - 좌표 중앙값이 서울 범위를 벗어난 분기는 좌표 오류로 판단한다.
# ============================================================

ACADEMY_GROUPS = {
    "입시교과": ["P10501"],
    "예체능": ["P10601", "P10609", "P10611", "P10613"],
    "외국어": ["P10615"],
}

GAP_RATIO = 0.7

SEOUL_LAT = (37.41, 37.72)
SEOUL_LON = (126.76, 127.19)


snapshot_counts = (
    store_edu
    .assign(
        입시교과=store_edu["상권업종소분류코드"].isin(ACADEMY_GROUPS["입시교과"]),
        예체능=store_edu["상권업종소분류코드"].isin(ACADEMY_GROUPS["예체능"]),
        외국어=store_edu["상권업종소분류코드"].isin(ACADEMY_GROUPS["외국어"]),
    )
    .groupby("기준월")
    .agg(
        전체업소수=("전체업소수", "first"),
        입시교과=("입시교과", "sum"),
        예체능=("예체능", "sum"),
        외국어=("외국어", "sum"),
        위도_중앙값=("위도", "median"),
        경도_중앙값=("경도", "median"),
    )
)


snapshot_counts["수집정상"] = (
    snapshot_counts["입시교과"]
    >= snapshot_counts["입시교과"].median() * GAP_RATIO
)

snapshot_counts["좌표정상"] = (
    snapshot_counts["위도_중앙값"].between(*SEOUL_LAT)
    & snapshot_counts["경도_중앙값"].between(*SEOUL_LON)
)


snapshot_counts["정상여부"] = (
    snapshot_counts["수집정상"]
    & snapshot_counts["좌표정상"]
)


display(snapshot_counts)


gap_months = snapshot_counts.index[~snapshot_counts["정상여부"]].tolist()

print("결손 분기:", gap_months)

assert gap_months == [
    "202303", "202306", "202309", "202312",
    "202403", "202406", "202409",
    "202503",
]

assert snapshot_counts.loc["202503", "수집정상"]
assert not snapshot_counts.loc["202503", "좌표정상"]

,전체업소수,입시교과,예체능,외국어,위도_중앙값,경도_중앙값,수집정상,좌표정상,정상여부
기준월,,,,,,,,,
202003,479384,11489,7766,900,37.533276,127.015089,True,True,True
202006,484765,11816,7983,934,37.533539,127.015083,True,True,True
202009,491092,12280,8221,972,37.533576,127.015083,True,True,True
202012,494343,12703,8425,1013,37.533756,127.015119,True,True,True
202103,502601,13139,8676,1058,37.533645,127.015196,True,True,True
202106,507529,13527,8958,1105,37.533857,127.015195,True,True,True
202109,514889,13924,9190,1154,37.533857,127.015089,True,True,True
202112,517364,14253,9356,1183,37.534013,127.015072,True,True,True
202203,524728,14619,9626,1238,37.534075,127.014947,True,True,True


결손 분기: ['202303', '202306', '202309', '202312', '202403', '202406', '202409', '202503']


In [4]:
# ============================================================
# 거래별 사용 스냅샷 결정
# - 스냅샷은 기준월 + 3개월부터 사용 가능하다고 본다.
#   (공개 시차: 2026-06-30 기준 파일이 2026-08-05 공개 → +2개월이면 8월 초 거래가 공개 전 데이터를 사용)
# - 거래월 시점에 사용 가능한 정상 스냅샷 중 가장 최신 스냅샷을 연결한다.
# - 학원스냅샷_경과개월로 사용한 데이터가 얼마나 오래됐는지 기록한다.
# ============================================================

RELEASE_LAG_MONTHS = 3


def to_month_index(yyyymm):
    yyyymm = pd.Series(yyyymm).astype(int)
    return (yyyymm // 100) * 12 + (yyyymm % 100) - 1


snapshots = (
    snapshot_counts[snapshot_counts["정상여부"]]
    .reset_index()[["기준월"]]
)

snapshots["기준월_idx"] = to_month_index(snapshots["기준월"]).to_numpy()
snapshots["사용가능_idx"] = snapshots["기준월_idx"] + RELEASE_LAG_MONTHS


deals = (
    pd.read_parquet(
        PROCESSED_DIR / "df_final_v1.parquet",
        columns=[
            "transaction_id",
            "단지일련번호",
            "단지명",
            "시군구명",
            "위도",
            "경도",
            "거래연월",
        ],
    )
    .astype(
        {
            "거래연월": int,
            "단지명": str,
            "시군구명": str,
        }
    )
)

deals["거래월_idx"] = to_month_index(deals["거래연월"]).to_numpy()


deal_months = (
    deals[["거래연월", "거래월_idx"]]
    .drop_duplicates()
    .sort_values("거래월_idx")
)

deal_months = pd.merge_asof(
    deal_months,
    snapshots.sort_values("사용가능_idx"),
    left_on="거래월_idx",
    right_on="사용가능_idx",
    direction="backward",
)

deal_months["학원스냅샷_경과개월"] = (
    deal_months["거래월_idx"]
    - deal_months["기준월_idx"]
)


assert deal_months["기준월"].notna().all()
assert (deal_months["학원스냅샷_경과개월"] >= RELEASE_LAG_MONTHS).all()
assert not deal_months["기준월"].isin(gap_months).any()


display(
    deal_months
    .groupby("기준월")
    .agg(
        거래월_시작=("거래연월", "min"),
        거래월_끝=("거래연월", "max"),
        최대경과개월=("학원스냅샷_경과개월", "max"),
    )
)

,거래월_시작,거래월_끝,최대경과개월
기준월,,,
202103,202107,202108,5
202106,202109,202111,5
202109,202112,202202,5
202112,202203,202205,5
202203,202206,202208,5
202206,202209,202211,5
202209,202212,202302,5
202212,202303,202502,26
202412,202503,202508,8


In [5]:
# ============================================================
# 학원가 정의 (2020-12 스냅샷, 학습 기간 이전)
# - 입시·교과학원 좌표에 DBSCAN(반경 250m, 최소 80개)을 적용해 밀집 지역을 찾는다.
# - 학원 수 250개 이상인 군집을 '3대 학원가'(대치·중계·목동)로 구분한다.
# - 학원가 이름은 군집 내 최빈 자치구·행정동으로 붙인다.
# ============================================================

CLUSTER_SNAPSHOT = "202012"
CLUSTER_EPS_M = 250
CLUSTER_MIN_SAMPLES = 80
MAJOR_CLUSTER_SIZE = 250


cluster_base = store_edu[
    (store_edu["기준월"] == CLUSTER_SNAPSHOT)
    & store_edu["상권업종소분류코드"].isin(ACADEMY_GROUPS["입시교과"])
].copy()

cluster_base["군집"] = DBSCAN(
    eps=CLUSTER_EPS_M / EARTH_RADIUS_M,
    min_samples=CLUSTER_MIN_SAMPLES,
    metric="haversine",
    algorithm="ball_tree",
).fit_predict(
    np.radians(cluster_base[["위도", "경도"]].to_numpy())
)


academy_clusters = (
    cluster_base[cluster_base["군집"] >= 0]
    .groupby("군집")
    .agg(
        학원수=("군집", "size"),
        위도=("위도", "mean"),
        경도=("경도", "mean"),
        자치구=("시군구명", lambda s: s.mode()[0]),
        행정동=("행정동명", lambda s: s.mode()[0]),
    )
    .sort_values("학원수", ascending=False)
    .reset_index(drop=True)
)

academy_clusters["학원가명"] = (
    academy_clusters["자치구"]
    + " "
    + academy_clusters["행정동"]
)

academy_clusters["3대학원가"] = (
    academy_clusters["학원수"] >= MAJOR_CLUSTER_SIZE
)


display(academy_clusters)

assert academy_clusters.loc[academy_clusters["3대학원가"], "자치구"].tolist() == [
    "강남구", "노원구", "양천구",
]

,학원수,위도,경도,자치구,행정동,학원가명,3대학원가
0,888,37.499650,127.057946,강남구,대치4동,강남구 대치4동,True
1,401,37.650399,127.076877,노원구,중계본동,노원구 중계본동,True
2,306,37.536235,126.879229,양천구,목5동,양천구 목5동,True
3,129,37.501724,127.010617,서초구,반포4동,서초구 반포4동,False
4,127,37.541326,127.095228,광진구,구의3동,광진구 구의3동,False
5,125,37.603481,127.026493,성북구,돈암1동,성북구 돈암1동,False
6,123,37.553070,127.154839,강동구,명일2동,강동구 명일2동,False
7,116,37.510462,127.125251,송파구,방이1동,송파구 방이1동,False
8,93,37.493643,127.061349,강남구,대치1동,강남구 대치1동,False
9,87,37.517411,126.865313,양천구,신정6동,양천구 신정6동,False


In [6]:
# ============================================================
# 단지 좌표별 학원 Feature 생성
# - 스냅샷별: 반경 500m·1km 입시·교과학원 수, 1km 예체능·외국어 학원 수
# - 정적: 최근접 학원가·3대 학원가 거리 (2020-12 기준 위치)
# ============================================================

RADIUS_RULES = [
    ("입시교과", 500, "반경500m_입시교과학원수"),
    ("입시교과", 1000, "반경1km_입시교과학원수"),
    ("예체능", 1000, "반경1km_예체능학원수"),
    ("외국어", 1000, "반경1km_외국어학원수"),
]


locations = (
    deals[["위도", "경도"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

location_radians = np.radians(
    locations[["위도", "경도"]].to_numpy()
)


def build_tree(df):
    return BallTree(
        np.radians(df[["위도", "경도"]].to_numpy()),
        metric="haversine",
    )


def build_snapshot_features(snapshot_month):

    snapshot = store_edu[store_edu["기준월"] == snapshot_month]

    features = locations.copy()
    features["기준월"] = snapshot_month

    for group, radius_m, name in RADIUS_RULES:

        academies = snapshot[
            snapshot["상권업종소분류코드"].isin(ACADEMY_GROUPS[group])
        ]

        features[name] = (
            build_tree(academies)
            .query_radius(
                location_radians,
                r=radius_m / EARTH_RADIUS_M,
                count_only=True,
            )
        )

    return features


snapshot_features = pd.concat(
    [
        build_snapshot_features(month)
        for month in sorted(deal_months["기준월"].unique())
    ],
    ignore_index=True,
)


static_features = locations.copy()

for label, clusters in [
    ("학원가", academy_clusters),
    ("3대학원가", academy_clusters[academy_clusters["3대학원가"]]),
]:

    clusters = clusters.reset_index(drop=True)

    distances, indices = build_tree(clusters).query(location_radians, k=1)

    static_features[f"최근접{label}_거리_m"] = distances[:, 0] * EARTH_RADIUS_M
    static_features[f"최근접{label}명"] = clusters["학원가명"].to_numpy()[indices[:, 0]]


print("스냅샷 × 좌표 Feature:", snapshot_features.shape)
print("정적 Feature:", static_features.shape)

display(
    snapshot_features
    .groupby("기준월")[[name for _, _, name in RADIUS_RULES]]
    .mean()
    .round(1)
)

스냅샷 × 좌표 Feature: (101660, 7)
정적 Feature: (7820, 6)


,반경500m_입시교과학원수,반경1km_입시교과학원수,반경1km_예체능학원수,반경1km_외국어학원수
기준월,,,,
202103,30.8,105.6,70.2,8.1
202106,31.7,108.6,72.4,8.4
202109,32.6,111.7,74.2,8.8
202112,33.3,114.4,75.5,9.0
202203,34.2,117.2,77.6,9.4
202206,34.9,119.8,79.3,9.6
202209,35.7,122.3,80.9,9.9
202212,36.0,123.6,81.8,10.0
202412,36.2,125.5,79.8,10.1


In [7]:
# ============================================================
# 거래 단위 학원 Feature 생성 및 검증
# - 거래월 → 사용 스냅샷 → 좌표별 Feature 순서로 연결한다.
# - 결손 분기를 건너뛰었으므로 거래 시기별 평균 학원 수에 급락이 없어야 한다.
# ============================================================

df_fe_academy_v2 = (
    deals[["transaction_id", "위도", "경도", "거래연월"]]
    .merge(
        deal_months[["거래연월", "기준월", "학원스냅샷_경과개월"]],
        on="거래연월",
        how="left",
        validate="many_to_one",
    )
    .merge(
        snapshot_features,
        on=["위도", "경도", "기준월"],
        how="left",
        validate="many_to_one",
    )
    .merge(
        static_features,
        on=["위도", "경도"],
        how="left",
        validate="many_to_one",
    )
    .rename(columns={"기준월": "학원스냅샷기준월"})
)


ACADEMY_V2_FEATURES = [
    "반경500m_입시교과학원수",
    "반경1km_입시교과학원수",
    "반경1km_예체능학원수",
    "반경1km_외국어학원수",
    "최근접학원가_거리_m",
    "최근접3대학원가_거리_m",
]

# 결손 분기를 건너뛰면 2023~2024 거래만 경과개월이 커져(최대 25개월) 시기 표시처럼 작동한다.
# 모델 Feature가 아닌 데이터 신선도 메타데이터로만 보관한다.
ACADEMY_V2_META_COLS = [
    "학원스냅샷기준월",
    "학원스냅샷_경과개월",
]

ACADEMY_V2_DISPLAY_COLS = [
    "최근접학원가명",
    "최근접3대학원가명",
]


print("거래 단위 학원 Feature:", df_fe_academy_v2.shape)

assert len(df_fe_academy_v2) == len(deals)
assert df_fe_academy_v2["transaction_id"].is_unique
assert df_fe_academy_v2[ACADEMY_V2_FEATURES + ACADEMY_V2_DISPLAY_COLS + ACADEMY_V2_META_COLS].isna().sum().sum() == 0


trend = (
    df_fe_academy_v2
    .assign(거래분기=(df_fe_academy_v2["거래연월"] // 100).astype(str)
            + "Q" + ((df_fe_academy_v2["거래연월"] % 100 - 1) // 3 + 1).astype(str))
    .groupby("거래분기")
    .agg(
        사용스냅샷=("학원스냅샷기준월", "max"),
        평균_반경1km_입시교과=("반경1km_입시교과학원수", "mean"),
    )
    .round(1)
)

display(trend)

# 직전 분기 대비 30% 이상 급락이 없어야 한다. (결손 분기 차단 확인)
assert (trend["평균_반경1km_입시교과"].pct_change().dropna() > -0.3).all()

거래 단위 학원 Feature: (234302, 14)


,사용스냅샷,평균_반경1km_입시교과
거래분기,,
2021Q3,202106,108.8
2021Q4,202109,111.3
2022Q1,202112,113.1
2022Q2,202203,115.6
2022Q3,202206,120.9
2022Q4,202209,126.5
2023Q1,202212,140.3
2023Q2,202212,146.0
2023Q3,202212,140.4


## 서울시 학원 교습소정보 (웹앱 표시용)

현재 개원 중인 학원·교습소 명부로 단지 상세 화면의 사교육 카드를 만든다.
도로명주소를 Kakao 주소검색으로 좌표 변환하고, 결과는 캐시에 저장해 재실행 시 API를 다시 호출하지 않는다.

In [8]:
# ============================================================
# 학원 교습소정보 로드 및 정리
# - 분야 '입시.검정 및 보습'을 입시·보습으로 구분한다.
# - 정원합계는 10,000명 이상 값을 입력 오류로 보고 결측 처리한다. (최대 999,999,990명)
# - 정원합계 0은 미입력으로 보고 결측 처리한다. (약 19%)
# - 행정구역명 결측은 도로명주소의 자치구로 채운다.
# ============================================================

CAPACITY_MAX = 10_000


registry = pd.read_csv(
    EXTERNAL_DIR / "서울시 학원 교습소정보.csv",
    encoding="cp949",
    dtype=str,
)

registry["도로명주소"] = registry["도로명주소"].str.strip()
registry["입시보습"] = registry["분야명"] == "입시.검정 및 보습"

registry["정원합계"] = pd.to_numeric(registry["정원합계"], errors="coerce")
registry.loc[registry["정원합계"] >= CAPACITY_MAX, "정원합계"] = np.nan
registry.loc[registry["정원합계"] == 0, "정원합계"] = np.nan

registry["행정구역명"] = registry["행정구역명"].fillna(
    registry["도로명주소"].str.split().str[1]
)

assert registry["행정구역명"].isna().sum() == 0
assert registry["행정구역명"].nunique() == 25


print("학원·교습소:", registry["학원/교습소"].value_counts().to_dict())
print("입시·보습:", registry["입시보습"].sum())
print("정원 결측 (미입력·이상값):", registry["정원합계"].isna().sum())

학원·교습소: {'학원': 15299, '교습소': 10228}
입시·보습: 14160
정원 결측 (미입력·이상값): 4961


In [9]:
# ============================================================
# 도로명주소 지오코딩 (Kakao 주소검색, 캐시 사용)
# - V1 지오코딩 방식과 동일하게 검색주소·위도·경도를 CSV 캐시로 관리한다.
# - 캐시에 없는 주소만 API를 호출한다.
# ============================================================

load_dotenv(PROJECT_DIR / ".env")

KAKAO_API_KEY = os.getenv("KAKAO_REST_API_KEY")

ACADEMY_GEOCODE_CACHE = EXTERNAL_DIR / "academy_geocode_cache.csv"
GEOCODE_COLUMNS = ["검색주소", "위도", "경도"]


def geocode_address(query):

    for attempt in range(3):

        try:
            response = requests.get(
                "https://dapi.kakao.com/v2/local/search/address.json",
                headers={"Authorization": f"KakaoAK {KAKAO_API_KEY}"},
                params={"query": query},
                timeout=5,
            )

            if response.status_code == 429:
                time.sleep(2 ** attempt)
                continue

            response.raise_for_status()
            documents = response.json().get("documents", [])

            if not documents:
                return None, None

            return float(documents[0]["y"]), float(documents[0]["x"])

        except requests.RequestException:
            time.sleep(1 + attempt)

    return None, None


geocode_cache = (
    pd.read_csv(ACADEMY_GEOCODE_CACHE, encoding="utf-8-sig")[GEOCODE_COLUMNS]
    if ACADEMY_GEOCODE_CACHE.exists()
    else pd.DataFrame(columns=GEOCODE_COLUMNS)
)

todo = sorted(
    set(registry["도로명주소"].dropna())
    - set(geocode_cache["검색주소"])
)

print("캐시 주소:", len(geocode_cache), "/ 신규 호출:", len(todo))


if todo:

    new_rows = []

    for address in todo:
        lat, lon = geocode_address(address)
        new_rows.append({"검색주소": address, "위도": lat, "경도": lon})
        time.sleep(0.03)

    geocode_cache = (
        pd.concat([geocode_cache, pd.DataFrame(new_rows)], ignore_index=True)
        .drop_duplicates("검색주소", keep="last")
    )

    geocode_cache.to_csv(ACADEMY_GEOCODE_CACHE, index=False, encoding="utf-8-sig")


registry = registry.merge(
    geocode_cache.rename(columns={"검색주소": "도로명주소"}),
    on="도로명주소",
    how="left",
    validate="many_to_one",
)


print("좌표 변환 성공률:", round(registry["위도"].notna().mean() * 100, 2), "%")

캐시 주소: 12416 / 신규 호출: 0
좌표 변환 성공률: 99.91 %


In [10]:
# ============================================================
# 학원 교습소정보 ↔ 상가정보 교차 검증
# - 자치구별 입시·보습 학원·교습소 수(현재 명부)와
#   최신 상가정보(2026-03) 입시·교과학원 수가 비슷한 분포인지 확인한다.
# ============================================================

registry_gu = (
    registry[registry["입시보습"]]
    .groupby("행정구역명")
    .size()
    .rename("교습소정보_입시보습")
)

store_gu = (
    store_edu[
        (store_edu["기준월"] == snapshot_counts.index.max())
        & store_edu["상권업종소분류코드"].isin(ACADEMY_GROUPS["입시교과"])
    ]
    .groupby("시군구명")
    .size()
    .rename("상가정보_입시교과")
)


gu_compare = pd.concat([registry_gu, store_gu], axis=1).sort_values(
    "교습소정보_입시보습", ascending=False
)

display(gu_compare.head(10))

gu_corr = gu_compare.corr().iloc[0, 1]
print("자치구별 상관계수:", round(gu_corr, 3))

assert gu_corr > 0.9

,교습소정보_입시보습,상가정보_입시교과
강남구,1939,2313
양천구,1414,1443
송파구,1071,1221
서초구,921,1235
노원구,881,1057
강동구,853,961
강서구,821,912
은평구,649,679
성북구,605,775
마포구,514,768


자치구별 상관계수: 0.987


In [11]:
# ============================================================
# 웹앱 표시용 현재 사교육 Feature (단지 좌표 단위)
# - 반경 1km 입시·보습 학원·교습소 수와 정원 합계 (정원을 입력한 곳만 합산)
# - 정원 300명 이상 입시·보습 학원을 대형학원으로 집계한다.
# - 현재 시점 정보이므로 모델 Feature로 사용하지 않는다.
# ============================================================

LARGE_ACADEMY_CAPACITY = 300


current = registry[
    registry["입시보습"]
    & registry["위도"].notna()
].reset_index(drop=True)

neighbor_indices = (
    build_tree(current)
    .query_radius(location_radians, r=1000 / EARTH_RADIUS_M)
)

capacity = current["정원합계"].fillna(0).to_numpy()
is_large = (current["정원합계"] >= LARGE_ACADEMY_CAPACITY).to_numpy()


academy_display_current = locations.copy()

academy_display_current["현재_반경1km_입시보습수"] = [len(idx) for idx in neighbor_indices]
academy_display_current["현재_반경1km_입시보습정원"] = [capacity[idx].sum() for idx in neighbor_indices]
academy_display_current["현재_반경1km_대형입시학원수"] = [is_large[idx].sum() for idx in neighbor_indices]

academy_display_current = academy_display_current.merge(
    static_features,
    on=["위도", "경도"],
    validate="one_to_one",
)


display(
    academy_display_current
    .describe()
    .T
    .round(1)
)

,count,mean,std,min,25%,50%,75%,max
위도,7820.0,37.5,0.0,37.4,37.5,37.5,37.6,37.7
경도,7820.0,127.0,0.1,126.8,126.9,127.0,127.1,127.2
현재_반경1km_입시보습수,7820.0,112.6,120.1,0.0,49.0,87.0,135.0,1287.0
현재_반경1km_입시보습정원,7820.0,16897.3,32114.4,0.0,5293.8,9992.0,18228.2,404284.0
현재_반경1km_대형입시학원수,7820.0,13.8,25.5,0.0,3.0,7.0,17.0,308.0
최근접학원가_거리_m,7820.0,3588.8,2093.0,19.9,2021.4,3226.7,4778.6,10625.6
최근접3대학원가_거리_m,7820.0,5577.0,2427.7,44.9,3660.9,5588.5,7501.0,13036.6


In [12]:
# ============================================================
# B2C 웹앱 사교육 카드 예시
# - 거래가 가장 많은 단지와 대치 학원가에 가장 가까운 단지를 비교한다.
# - 단지명은 중복('현대' 등)이 많으므로 단지일련번호로 단지를 식별한다.
# ============================================================

complexes = (
    deals
    .groupby(["단지일련번호", "단지명", "시군구명"])
    .agg(
        거래수=("transaction_id", "size"),
        위도=("위도", "first"),
        경도=("경도", "first"),
    )
    .reset_index()
    .merge(
        academy_display_current,
        on=["위도", "경도"],
        validate="many_to_one",
    )
)


def print_academy_card(card):

    print(f"🏢 {card['단지명']} ({card['시군구명']})")
    print(
        f"📖 반경 1km 입시·보습 학원·교습소 {card['현재_반경1km_입시보습수']:,}곳 "
        f"· 정원 {card['현재_반경1km_입시보습정원']:,.0f}명(입력 기준) "
        f"· 대형학원 {card['현재_반경1km_대형입시학원수']}곳"
    )
    print(
        f"📍 가까운 학원가: {card['최근접학원가명']} {card['최근접학원가_거리_m'] / 1000:.1f}km "
        f"· 3대 학원가({card['최근접3대학원가명']}) {card['최근접3대학원가_거리_m'] / 1000:.1f}km\n"
    )


daechi = academy_clusters[academy_clusters["학원가명"] == "강남구 대치4동"].iloc[0]

complexes["대치학원가_거리"] = np.hypot(
    complexes["위도"] - daechi["위도"],
    complexes["경도"] - daechi["경도"],
)

print_academy_card(complexes.sort_values("거래수", ascending=False).iloc[0])
print_academy_card(complexes.sort_values("대치학원가_거리").iloc[0])

🏢 헬리오시티 (송파구)
📖 반경 1km 입시·보습 학원·교습소 94곳 · 정원 14,814명(입력 기준) · 대형학원 17곳
📍 가까운 학원가: 송파구 방이1동 2.2km · 3대 학원가(강남구 대치4동) 4.3km

🏢 세연파크뷰 (강남구)
📖 반경 1km 입시·보습 학원·교습소 1,276곳 · 정원 401,049명(입력 기준) · 대형학원 305곳
📍 가까운 학원가: 강남구 대치4동 0.1km · 3대 학원가(강남구 대치4동) 0.1km



In [13]:
# ============================================================
# 학원 Feature 저장 + 최종 학교환경 파일 생성
# - 학원 Feature(거래 단위), 학원가 목록, 웹앱 표시용 현재 사교육 테이블을 저장한다.
# - 02b 학교 Feature와 transaction_id로 합쳐 school_environment_v2.parquet을 만든다.
# ============================================================

df_fe_academy_v2 = df_fe_academy_v2[
    ["transaction_id"]
    + ACADEMY_V2_META_COLS
    + ACADEMY_V2_FEATURES
    + ACADEMY_V2_DISPLAY_COLS
]

df_fe_academy_v2.to_parquet(PROCESSED_DIR / "df_fe_academy_v2.parquet", index=False)
academy_clusters.to_parquet(ACADEMY_DIR / "academy_clusters_202012.parquet", index=False)
academy_display_current.to_parquet(ACADEMY_DIR / "academy_display_current.parquet", index=False)


school_features = pd.read_parquet(PROCESSED_DIR / "df_fe_school_v2.parquet")

school_environment_v2 = school_features.merge(
    df_fe_academy_v2,
    on="transaction_id",
    how="inner",
    validate="one_to_one",
)

assert len(school_environment_v2) == len(deals)

SCHOOL_ENVIRONMENT_PATH = PROCESSED_DIR / "school_environment_v2.parquet"
school_environment_v2.to_parquet(SCHOOL_ENVIRONMENT_PATH, index=False)


non_feature_cols = (
    ["transaction_id", "학교기준연도"]
    + ACADEMY_V2_META_COLS
    + [col for col in school_environment_v2.columns if col.endswith("명")]
)

model_feature_cols = [
    col for col in school_environment_v2.columns
    if col not in non_feature_cols
]

assert set(ACADEMY_V2_FEATURES) <= set(model_feature_cols)
assert "학원스냅샷_경과개월" not in model_feature_cols

print(f"✅ {SCHOOL_ENVIRONMENT_PATH.name}: {school_environment_v2.shape}")
print("모델 Feature 수:", len(model_feature_cols))
print("Key·시점·메타·표시용 컬럼:", non_feature_cols)

✅ school_environment_v2.parquet: (234302, 37)
모델 Feature 수: 29
Key·시점·메타·표시용 컬럼: ['transaction_id', '학교기준연도', '학원스냅샷기준월', '학원스냅샷_경과개월', '최근접초_학교명', '최근접중_학교명', '최근접학원가명', '최근접3대학원가명']
